# 🎮 AI Trivia Quiz — Step 3: Game Logic
We build the brain of the game — scoring, timer, state management.
No UI yet, just pure Python logic we can test here.

In [1]:
import time

# ── Game Configuration ────────────────────────────────────────────────────────
GAME_CONFIG = {
    'num_questions'   : 10,    # questions per game
    'time_per_question': 20,   # seconds per question
    'points': {
        'Easy'  : 10,
        'Medium': 20,
        'Hard'  : 30,
    },
    'time_bonus': True,        # extra points for answering fast
}

print('Game config loaded:')
for k, v in GAME_CONFIG.items():
    print(f'  {k}: {v}')

Game config loaded:
  num_questions: 10
  time_per_question: 20
  points: {'Easy': 10, 'Medium': 20, 'Hard': 30}
  time_bonus: True


In [2]:
def calculate_score(difficulty: str, time_taken: float, time_limit: int) -> int:
    """
    Calculate points for a correct answer.
    Base points by difficulty + time bonus for fast answers.
    """
    base = GAME_CONFIG['points'][difficulty]

    # Time bonus: up to 50% extra if answered in first half of time
    time_remaining = max(0, time_limit - time_taken)
    time_bonus = int((time_remaining / time_limit) * base * 0.5)

    return base + time_bonus


def get_grade(score: int, total_questions: int, difficulty: str) -> dict:
    """
    Return grade, emoji and message based on score percentage.
    """
    max_score = GAME_CONFIG['points'][difficulty] * total_questions
    pct = (score / max_score * 100) if max_score > 0 else 0

    if pct >= 90:
        return {'grade': 'S', 'emoji': '🏆', 'msg': 'Legendary! You are a trivia master!',  'color': '#FFD700'}
    elif pct >= 75:
        return {'grade': 'A', 'emoji': '🥇', 'msg': 'Excellent! You really know your stuff!', 'color': '#4CAF50'}
    elif pct >= 60:
        return {'grade': 'B', 'emoji': '🥈', 'msg': 'Good job! Solid performance!',           'color': '#2196F3'}
    elif pct >= 40:
        return {'grade': 'C', 'emoji': '🥉', 'msg': 'Not bad! Keep practicing!',              'color': '#FF9800'}
    else:
        return {'grade': 'D', 'emoji': '😅', 'msg': 'Better luck next time! Try again!',      'color': '#F44336'}


def init_game_state(questions: list, category: str, difficulty: str) -> dict:
    """
    Initialize a fresh game state dictionary.
    This is what Streamlit session_state will store.
    """
    return {
        'questions'       : questions,
        'category'        : category,
        'difficulty'      : difficulty,
        'current_index'   : 0,
        'score'           : 0,
        'correct_count'   : 0,
        'answers_given'   : [],   # list of {question, chosen, correct, points, time_taken}
        'question_start'  : time.time(),
        'game_phase'      : 'playing',  # 'playing' | 'result'
    }


def submit_answer(state: dict, chosen_option: str) -> dict:
    """
    Process the player's answer. Updates state in-place.
    Returns a result dict for the UI to display feedback.
    """
    q          = state['questions'][state['current_index']]
    time_taken = time.time() - state['question_start']
    is_correct = (chosen_option == q['answer'])
    points     = 0

    if is_correct:
        points = calculate_score(state['difficulty'], time_taken,
                                 GAME_CONFIG['time_per_question'])
        state['score']         += points
        state['correct_count'] += 1

    state['answers_given'].append({
        'question'   : q['question'],
        'chosen'     : chosen_option,
        'correct'    : q['answer'],
        'explanation': q['explanation'],
        'is_correct' : is_correct,
        'points'     : points,
        'time_taken' : round(time_taken, 1),
    })

    # Advance to next question or end game
    state['current_index']  += 1
    state['question_start']  = time.time()

    if state['current_index'] >= len(state['questions']):
        state['game_phase'] = 'result'

    return {'is_correct': is_correct, 'points': points, 'correct_answer': q['answer']}


print('✅ Game logic functions defined!')

✅ Game logic functions defined!


In [3]:
# TEST the game logic with fake questions
fake_questions = [
    {
        'question': 'What is the capital of France?',
        'options': ['A) London', 'B) Berlin', 'C) Paris', 'D) Madrid'],
        'answer': 'C) Paris',
        'explanation': 'Paris has been the capital of France since 987 AD.'
    },
    {
        'question': 'What is 2 + 2?',
        'options': ['A) 3', 'B) 4', 'C) 5', 'D) 22'],
        'answer': 'B) 4',
        'explanation': 'Basic arithmetic: 2 + 2 = 4.'
    },
]

# Start game
state = init_game_state(fake_questions, '🌍 General Knowledge', 'Medium')
print('Game started! Phase:', state['game_phase'])
print('Current Q:', state['questions'][state['current_index']]['question'])

# Answer Q1 correctly and fast
time.sleep(2)  # simulate 2 seconds thinking
result1 = submit_answer(state, 'C) Paris')
print(f'\nQ1 → Correct: {result1["is_correct"]} | Points earned: {result1["points"]}')
print('Score so far:', state['score'])

# Answer Q2 wrong
result2 = submit_answer(state, 'A) 3')
print(f'Q2 → Correct: {result2["is_correct"]} | Points earned: {result2["points"]}')
print('Final score:', state['score'])
print('Game phase:', state['game_phase'])

# Get grade
grade = get_grade(state['score'], len(fake_questions), 'Medium')
print(f'\nGrade: {grade["emoji"]} {grade["grade"]} — {grade["msg"]}')

Game started! Phase: playing
Current Q: What is the capital of France?

Q1 → Correct: True | Points earned: 28
Score so far: 28
Q2 → Correct: False | Points earned: 0
Final score: 28
Game phase: result

Grade: 🥈 B — Good job! Solid performance!


In [5]:
# Save game_logic.py for the Streamlit app

code = '''import time

GAME_CONFIG = {
    "num_questions"    : 10,
    "time_per_question": 20,
    "points": {"Easy": 10, "Medium": 20, "Hard": 30},
    "time_bonus": True,
}

def calculate_score(difficulty, time_taken, time_limit):
    base = GAME_CONFIG["points"][difficulty]
    time_remaining = max(0, time_limit - time_taken)
    time_bonus = int((time_remaining / time_limit) * base * 0.5)
    return base + time_bonus

def get_grade(score, total_questions, difficulty):
    max_score = GAME_CONFIG["points"][difficulty] * total_questions
    pct = (score / max_score * 100) if max_score > 0 else 0
    if pct >= 90: return {"grade": "S", "emoji": "🏆", "msg": "Legendary! You are a trivia master!",   "color": "#FFD700"}
    elif pct >= 75: return {"grade": "A", "emoji": "🥇", "msg": "Excellent! You really know your stuff!", "color": "#4CAF50"}
    elif pct >= 60: return {"grade": "B", "emoji": "🥈", "msg": "Good job! Solid performance!",           "color": "#2196F3"}
    elif pct >= 40: return {"grade": "C", "emoji": "🥉", "msg": "Not bad! Keep practicing!",              "color": "#FF9800"}
    else:           return {"grade": "D", "emoji": "😅", "msg": "Better luck next time! Try again!",      "color": "#F44336"}

def init_game_state(questions, category, difficulty):
    return {
        "questions"      : questions,
        "category"       : category,
        "difficulty"     : difficulty,
        "current_index"  : 0,
        "score"          : 0,
        "correct_count"  : 0,
        "answers_given"  : [],
        "question_start" : time.time(),
        "game_phase"     : "playing",
    }

def submit_answer(state, chosen_option):
    q          = state["questions"][state["current_index"]]
    time_taken = time.time() - state["question_start"]
    is_correct = (chosen_option == q["answer"])
    points     = 0
    if is_correct:
        points = calculate_score(state["difficulty"], time_taken, GAME_CONFIG["time_per_question"])
        state["score"]         += points
        state["correct_count"] += 1
    state["answers_given"].append({
        "question"   : q["question"],
        "chosen"     : chosen_option,
        "correct"    : q["answer"],
        "explanation": q["explanation"],
        "is_correct" : is_correct,
        "points"     : points,
        "time_taken" : round(time_taken, 1),
    })
    state["current_index"] += 1
    state["question_start"] = time.time()
    if state["current_index"] >= len(state["questions"]):
        state["game_phase"] = "result"
    return {"is_correct": is_correct, "points": points, "correct_answer": q["answer"]}
'''

with open('trivia_app/game_logic.py', 'w',encoding='utf-8') as f:
    f.write(code)

print('✅ trivia_app/game_logic.py saved!')

✅ trivia_app/game_logic.py saved!
